# Stable Merge Sort and Verification


```{contents}
:local:
:depth: 2
```


```{index} Prove the Combining Step Before Hiding It in Recursion
```



```{index} Prove the Combining Step Before Hiding It in Recursion
```

## Prove the Combining Step Before Hiding It in Recursion



Assume two input runs are already sorted under the same policy. To merge them, compare their first unread elements and choose the smaller. No later element in either run can be smaller than that run's first unread element, so the chosen value is safe as the next output. Recursion supplies sorted runs; it does not remove the need to justify this combining rule.

Merge [2, 5] with [1, 5, 7]. Choose 1 from the right, then 2 from the left. At the tie between the two 5 values, choosing the left one first preserves relative order when these runs came from adjacent parts of the original input. After one run is exhausted, copy the remaining suffix of the other; it is already sorted.

| Output so far | Left unread | Right unread |
| --- | --- | --- |
| [] | 2, 5 | 1, 5, 7 |
| [1] | 2, 5 | 5, 7 |
| [1, 2] | 5 | 5, 7 |
| [1, 2, 5-left] | Empty | 5-right, 7 |

The invariant combines ordering and preservation: output contains exactly the consumed input occurrences in sorted order, and its final element is no greater than the remaining candidates. Keeping only the ordering claim would not rule out accidentally duplicating one occurrence and losing another.

Range/buffer implementations add an indexing obligation. Source and destination ranges must refer to the intended segments; copying into the wrong offset can produce correct tiny whole-array results and fail on an interior range. Trace a merge whose lo is not zero, and include odd lengths. A reusable buffer can reduce allocation without changing the merge contract.

Guided verification: label equal invoice amounts with original IDs. Check that sorted order holds, every original ID occurs exactly once, and equal-key IDs retain their arrival order. Then compare a nonmutating sort that returns a fresh outer array with an in-place sort. Both may order values correctly, but only one satisfies a contract that preserves the caller's original array order.

For related explanation, see the further reading below.[^development-reading]


```{index} Combining Sorted Parts
```

## Combining Sorted Parts




```{index} Combining Sorted Parts
```

Merge sort depends on a combine step called **merge**. Given two sorted arrays, merge walks through both arrays and builds one sorted result.

In [ ]:
using System;
using System.Linq;

int[] left={1,4,4,9},right={2,4,10};
Console.WriteLine(string.Join(", ",StableSort.Merge(left,right,(a,b)=>a.CompareTo(b))));
Console.WriteLine($"empty merge length={StableSort.Merge(Array.Empty<int>(),Array.Empty<int>(),(a,b)=>a.CompareTo(b)).Length}");
public static class StableSort
{
    public sealed class Counts
    {
        public long Comparisons, Writes, Calls;
        public int PeakFrames;
    }
    // Inputs must already be sorted under compare. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, Comparison<T> compare)
    {
        ArgumentNullException.ThrowIfNull(left);
        ArgumentNullException.ThrowIfNull(right);
        ArgumentNullException.ThrowIfNull(compare);
        T[] result = new T[checked(left.Length + right.Length)];
        int i=0,j=0,k=0;
        while(i<left.Length && j<right.Length)
            result[k++]=compare(left[i],right[j])<=0?left[i++]:right[j++];
        while(i<left.Length)result[k++]=left[i++];
        while(j<right.Length)result[k++]=right[j++];
        return result;
    }
    // Fresh array even for empty/singleton input. Source order is unchanged.
    public static T[] Sort<T>(T[] input, Comparison<T> compare, Counts counts)
    {
        ArgumentNullException.ThrowIfNull(input);
        ArgumentNullException.ThrowIfNull(compare);
        ArgumentNullException.ThrowIfNull(counts);
        counts.Comparisons=counts.Writes=counts.Calls=0;counts.PeakFrames=0;
        T[] values=(T[])input.Clone();
        T[] buffer=new T[values.Length];
        SortRange(0,values.Length,1);
        return values;

        void SortRange(int start,int end,int depth)
        {
            counts.Calls++;counts.PeakFrames=Math.Max(counts.PeakFrames,depth);
            if(end-start<=1)return;
            int middle=start+(end-start)/2;
            SortRange(start,middle,depth+1);
            SortRange(middle,end,depth+1);
            int i=start,j=middle,k=start;
            while(i<middle && j<end)
            {
                counts.Comparisons++;
                buffer[k++]=compare(values[i],values[j])<=0?values[i++]:values[j++];
                counts.Writes++;
            }
            while(i<middle){buffer[k++]=values[i++];counts.Writes++;}
            while(j<end){buffer[k++]=values[j++];counts.Writes++;}
            for(int p=start;p<end;p++){values[p]=buffer[p];counts.Writes++;}
        }
    }
}


Merge requires two inputs sorted under the same comparison, and returns a fresh sorted array containing every input occurrence. At the main loop head, k=i+j and result[0,k) is the stable sorted merge of the consumed prefixes. The next smallest unconsumed item must be at one of the two fronts. Choosing the left front on equality preserves original order. Once one side is exhausted, copy the other remainder without further comparisons. For total N>0, merging takes Θ(N) time/output space and at most N−1 key comparisons; empty sides can need zero comparisons but still require copying.

**Common mistake.** Stopping when one side is exhausted and returning the result as it stands. The remaining run of the other side is dropped, and the output is shorter than the input while still looking sorted; the two tail-copy loops are not optional.


```{index} Merge Sort
```

## Merge Sort




```{index} Merge Sort
```

Merge sort divides the array into halves until each piece has zero or one item, then merges sorted pieces back together.

In [ ]:
using System;

int[] values = { 8, 3, 5, 1, 9, 2 };
int[] sorted = MergeSort(values);

Console.WriteLine(string.Join(", ", sorted));

int[] MergeSort(int[] values)
{
    if (values.Length <= 1)
    {
        return (int[])values.Clone();
    }

    int middle = values.Length / 2;
    int[] left = values[..middle];
    int[] right = values[middle..];

    return Merge(MergeSort(left), MergeSort(right));
}

int[] Merge(int[] left, int[] right)
{
    int[] result = new int[left.Length + right.Length];
    int i = 0;
    int j = 0;
    int k = 0;

    while (i < left.Length && j < right.Length)
    {
        result[k++] = left[i] <= right[j] ? left[i++] : right[j++];
    }

    while (i < left.Length)
    {
        result[k++] = left[i++];
    }

    while (j < right.Length)
    {
        result[k++] = right[j++];
    }

    return result;
}


The copy-returning contract leaves the input unchanged and returns a fresh array even for zero or one item. Slices allocate new arrays, making this version easy to read but allocation-heavy. T(n)=T(floor(n/2))+T(ceil(n/2))+Θ(n), so time is Θ(n log n) for n≥2. Stack depth is Θ(log n). The next section replaces slices with index ranges and a reusable buffer; it also verifies stability and preservation. Comparisons are only one part of sorting cost: copying still happens when a side is exhausted.

**Common mistake.** Returning the input array itself from the base case instead of a clone. A caller that sorts a one-element array then holds the same object twice, and the copy-returning contract the rest of the page relies on is silently broken for that length.


```{index} Design Checklist
```

## Design Checklist




```{index} Design Checklist
```

When designing a divide-and-conquer algorithm, ask:

1. What is the smallest case I can solve directly?
2. How do I divide the input?
3. Do I solve one subproblem or multiple subproblems?
4. How do I combine the answers?
5. What recurrence describes the cost?

(exercise)=
```{index} Exercise
```

## Checkpoint: Apply the Basic Merge Sort



Use this checkpoint to apply the basic combining and recursive steps. The later merge-front trace, shared-buffer implementation, and labeled-invoice checks develop a fuller verification sequence; finish that development before the final verification practice.



```{index} Exercise
```

Merge two sorted invoice batches, including duplicates and an empty side. Verify the exact expected sequence, not just increasing order.

In [ ]:
using System;
using System.Linq;

int[] left={1,3,3},right={2,3};
// Your code starts here.
/* TODO */
// Your code ends here.

In [1]:
using System;
using System.Linq;

// Solution
int[] left={1,3,3},right={2,3};
int[] result=StableSort.Merge(left,right,(a,b)=>a.CompareTo(b));
if(!result.SequenceEqual(new[]{1,2,3,3,3}))throw new Exception("Merge mismatch.");
Console.WriteLine(string.Join(", ",result));
Console.WriteLine(string.Join(", ",StableSort.Merge(left,Array.Empty<int>(),(a,b)=>a.CompareTo(b))));
public static class StableSort
{
    public sealed class Counts
    {
        public long Comparisons, Writes, Calls;
        public int PeakFrames;
    }
    // Inputs must already be sorted under compare. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, Comparison<T> compare)
    {
        ArgumentNullException.ThrowIfNull(left);
        ArgumentNullException.ThrowIfNull(right);
        ArgumentNullException.ThrowIfNull(compare);
        T[] result = new T[checked(left.Length + right.Length)];
        int i=0,j=0,k=0;
        while(i<left.Length && j<right.Length)
            result[k++]=compare(left[i],right[j])<=0?left[i++]:right[j++];
        while(i<left.Length)result[k++]=left[i++];
        while(j<right.Length)result[k++]=right[j++];
        return result;
    }
    // Fresh array even for empty/singleton input. Source order is unchanged.
    public static T[] Sort<T>(T[] input, Comparison<T> compare, Counts counts)
    {
        ArgumentNullException.ThrowIfNull(input);
        ArgumentNullException.ThrowIfNull(compare);
        ArgumentNullException.ThrowIfNull(counts);
        counts.Comparisons=counts.Writes=counts.Calls=0;counts.PeakFrames=0;
        T[] values=(T[])input.Clone();
        T[] buffer=new T[values.Length];
        SortRange(0,values.Length,1);
        return values;

        void SortRange(int start,int end,int depth)
        {
            counts.Calls++;counts.PeakFrames=Math.Max(counts.PeakFrames,depth);
            if(end-start<=1)return;
            int middle=start+(end-start)/2;
            SortRange(start,middle,depth+1);
            SortRange(middle,end,depth+1);
            int i=start,j=middle,k=start;
            while(i<middle && j<end)
            {
                counts.Comparisons++;
                buffer[k++]=compare(values[i],values[j])<=0?values[i++]:values[j++];
                counts.Writes++;
            }
            while(i<middle){buffer[k++]=values[i++];counts.Writes++;}
            while(j<end){buffer[k++]=values[j++];counts.Writes++;}
            for(int p=start;p<end;p++){values[p]=buffer[p];counts.Writes++;}
        }
    }
}


1, 2, 3, 3, 3
1, 3, 3


The output is `1, 2, 3, 3, 3` and `1, 3, 3`. All three 3s survived the merge, which is the preservation check the `SequenceEqual` guard enforces, and merging with an empty right side copied the left run through unchanged.


```{index} Merge Sort Lab
```

<span id="merge-sort-lab"></span>


A finance team sorts invoices by amount while retaining the arrival order of equal amounts. Build correctness checks before comparing counts. Every code cell includes its own setup and implementation.

Download {download}`the complete stable merge-sort source and reference checks <../../demos/22/StableSort.cs>` for a console project.


```{index} Trace Merge Fronts
```

## Trace Merge Fronts




```{index} Trace Merge Fronts
```

Track consumed indexes i and j and output index k. Equal keys take the left item first. Predict which side runs out, then trace both the comparison loop and the tail copies.

In [ ]:
using System;
using System.Linq;

int[] left={1,4,4},right={2,4,9};
int i=0,j=0,k=0;int[] result=new int[left.Length+right.Length];
while(i<left.Length && j<right.Length)
{
 Console.WriteLine($"i={i}, j={j}, k={k}; fronts={left[i]}/{right[j]}");
 result[k++]=left[i]<=right[j]?left[i++]:right[j++];
}
while(i<left.Length)result[k++]=left[i++];
while(j<right.Length)result[k++]=right[j++];
Console.WriteLine(string.Join(", ",result));

The invariant k=i+j and sorted consumed-prefix merge explains why no value disappears. The measure (left.Length−i)+(right.Length−j) decreases after every copy. Stability is relative to a comparison key; equal integer values cannot display record order, so use labeled invoices next.

**Common mistake.** Advancing both `i` and `j` after a comparison. Only the side whose element was copied moves; advancing the other skips an unread element, and the invariant `k = i + j` stops holding at that step.


```{index} Sort Ranges with One Buffer
```

## Sort Ranges with One Buffer




```{index} Sort Ranges with One Buffer
```

The public method clones the input once and allocates one reusable buffer. SortRange(start,end) sorts a half-open range. Sibling calls finish sequentially, then merge their sorted ranges into the matching buffer range and copy back. Counts reset at each Sort call.

In [ ]:
using System;
using System.Linq;

int[] values={7,2,6,3,1,5,4};
var counts=new StableSort.Counts();
int[] sorted=StableSort.Sort(values,(a,b)=>a.CompareTo(b),counts);
Console.WriteLine($"input: {string.Join(", ",values)}");
Console.WriteLine($"sorted: {string.Join(", ",sorted)}");
Console.WriteLine($"comparisons={counts.Comparisons}, writes={counts.Writes}, calls={counts.Calls}, peak={counts.PeakFrames}");
public static class StableSort
{
    public sealed class Counts
    {
        public long Comparisons, Writes, Calls;
        public int PeakFrames;
    }
    // Inputs must already be sorted under compare. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, Comparison<T> compare)
    {
        ArgumentNullException.ThrowIfNull(left);
        ArgumentNullException.ThrowIfNull(right);
        ArgumentNullException.ThrowIfNull(compare);
        T[] result = new T[checked(left.Length + right.Length)];
        int i=0,j=0,k=0;
        while(i<left.Length && j<right.Length)
            result[k++]=compare(left[i],right[j])<=0?left[i++]:right[j++];
        while(i<left.Length)result[k++]=left[i++];
        while(j<right.Length)result[k++]=right[j++];
        return result;
    }
    // Fresh array even for empty/singleton input. Source order is unchanged.
    public static T[] Sort<T>(T[] input, Comparison<T> compare, Counts counts)
    {
        ArgumentNullException.ThrowIfNull(input);
        ArgumentNullException.ThrowIfNull(compare);
        ArgumentNullException.ThrowIfNull(counts);
        counts.Comparisons=counts.Writes=counts.Calls=0;counts.PeakFrames=0;
        T[] values=(T[])input.Clone();
        T[] buffer=new T[values.Length];
        SortRange(0,values.Length,1);
        return values;

        void SortRange(int start,int end,int depth)
        {
            counts.Calls++;counts.PeakFrames=Math.Max(counts.PeakFrames,depth);
            if(end-start<=1)return;
            int middle=start+(end-start)/2;
            SortRange(start,middle,depth+1);
            SortRange(middle,end,depth+1);
            int i=start,j=middle,k=start;
            while(i<middle && j<end)
            {
                counts.Comparisons++;
                buffer[k++]=compare(values[i],values[j])<=0?values[i++]:values[j++];
                counts.Writes++;
            }
            while(i<middle){buffer[k++]=values[i++];counts.Writes++;}
            while(j<end){buffer[k++]=values[j++];counts.Writes++;}
            for(int p=start;p<end;p++){values[p]=buffer[p];counts.Writes++;}
        }
    }
}


Assume each recursive call correctly sorts its smaller range and preserves its items. The merge invariant establishes the same promise for the parent. Empty/singleton ranges return directly. Odd lengths are fully covered. The copied output and buffer use Θ(n) storage, and the stack adds Θ(log n); total auxiliary space is Θ(n). Across all levels there are Θ(n log n) writes, but cumulative writes are not peak memory. The original slice version also allocates across levels; separate total allocation volume from peak live storage.

**Common mistake.** Merging into the same array that is being read, in place, from the front. Writing `values[k]` overwrites an unread left element before it is compared; the separate buffer and the final copy-back are what make the single-array version correct.


```{index} Verify Stability with Labeled Invoices
```

## Verify Stability with Labeled Invoices




```{index} Verify Stability with Labeled Invoices
```

A stable sort preserves arrival order among equal amounts. Compare only the amount, and track identity separately. A consistent comparison must define an ordering; subtracting integer keys can overflow, so use CompareTo.

In [ ]:
using System;
using System.Linq;

var invoices=new[]{("A",200),("B",100),("C",200),("D",100),("E",200)};
var counts=new StableSort.Counts();
var sorted=StableSort.Sort(invoices,(a,b)=>a.Item2.CompareTo(b.Item2),counts);
Console.WriteLine(string.Join(", ",sorted.Select(x=>$"{x.Item1}:{x.Item2}")));
if(!sorted.Select(x=>x.Item1).SequenceEqual(new[]{"B","D","A","C","E"}))throw new Exception("Stability failed.");
Console.WriteLine("stable invoice order passed");
public static class StableSort
{
    public sealed class Counts
    {
        public long Comparisons, Writes, Calls;
        public int PeakFrames;
    }
    // Inputs must already be sorted under compare. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, Comparison<T> compare)
    {
        ArgumentNullException.ThrowIfNull(left);
        ArgumentNullException.ThrowIfNull(right);
        ArgumentNullException.ThrowIfNull(compare);
        T[] result = new T[checked(left.Length + right.Length)];
        int i=0,j=0,k=0;
        while(i<left.Length && j<right.Length)
            result[k++]=compare(left[i],right[j])<=0?left[i++]:right[j++];
        while(i<left.Length)result[k++]=left[i++];
        while(j<right.Length)result[k++]=right[j++];
        return result;
    }
    // Fresh array even for empty/singleton input. Source order is unchanged.
    public static T[] Sort<T>(T[] input, Comparison<T> compare, Counts counts)
    {
        ArgumentNullException.ThrowIfNull(input);
        ArgumentNullException.ThrowIfNull(compare);
        ArgumentNullException.ThrowIfNull(counts);
        counts.Comparisons=counts.Writes=counts.Calls=0;counts.PeakFrames=0;
        T[] values=(T[])input.Clone();
        T[] buffer=new T[values.Length];
        SortRange(0,values.Length,1);
        return values;

        void SortRange(int start,int end,int depth)
        {
            counts.Calls++;counts.PeakFrames=Math.Max(counts.PeakFrames,depth);
            if(end-start<=1)return;
            int middle=start+(end-start)/2;
            SortRange(start,middle,depth+1);
            SortRange(middle,end,depth+1);
            int i=start,j=middle,k=start;
            while(i<middle && j<end)
            {
                counts.Comparisons++;
                buffer[k++]=compare(values[i],values[j])<=0?values[i++]:values[j++];
                counts.Writes++;
            }
            while(i<middle){buffer[k++]=values[i++];counts.Writes++;}
            while(j<end){buffer[k++]=values[j++];counts.Writes++;}
            for(int p=start;p<end;p++){values[p]=buffer[p];counts.Writes++;}
        }
    }
}


Changing the merge comparison from ≤0 to <0 would let equal right items overtake equal left items. Stability matters when reports preserve earlier arrival order or apply successive sorts. It is a property of the algorithm and comparison policy, not merely of sorted numeric output.

**Common mistake.** Sorting by amount and then sorting the result again by ID with a non-stable algorithm to "fix" the tie order. The second sort destroys the amount order; a stable sort applied once, with ties left in arrival order, is the contract reports depend on.


```{index} Check Order, Preservation, and Copying
```

## Check Order, Preservation, and Copying




```{index} Check Order, Preservation, and Copying
```

Sortedness alone is insufficient: returning an empty array is sorted but loses every invoice. Compare against an independent sorted reference, check input preservation and array identity, and include empty, singleton, odd, duplicate, reverse, negative, and extreme cases.

In [ ]:
using System;
using System.Linq;

int[][] cases={Array.Empty<int>(),new[]{4},new[]{3,1,2},new[]{2,2,1,2},new[]{5,4,3,2,1},new[]{int.MaxValue,0,int.MinValue}};
foreach(int[] values in cases)
{
 int[] before=(int[])values.Clone();var counts=new StableSort.Counts();
 int[] actual=StableSort.Sort(values,(a,b)=>a.CompareTo(b),counts);
 if(!actual.SequenceEqual(before.OrderBy(x=>x)))throw new Exception("Order/preservation mismatch.");
 if(!values.SequenceEqual(before)||ReferenceEquals(values,actual))throw new Exception("Copy contract failed.");
 Console.WriteLine($"[{string.Join(",",values)}] -> [{string.Join(",",actual)}] verified");
}
public static class StableSort
{
    public sealed class Counts
    {
        public long Comparisons, Writes, Calls;
        public int PeakFrames;
    }
    // Inputs must already be sorted under compare. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, Comparison<T> compare)
    {
        ArgumentNullException.ThrowIfNull(left);
        ArgumentNullException.ThrowIfNull(right);
        ArgumentNullException.ThrowIfNull(compare);
        T[] result = new T[checked(left.Length + right.Length)];
        int i=0,j=0,k=0;
        while(i<left.Length && j<right.Length)
            result[k++]=compare(left[i],right[j])<=0?left[i++]:right[j++];
        while(i<left.Length)result[k++]=left[i++];
        while(j<right.Length)result[k++]=right[j++];
        return result;
    }
    // Fresh array even for empty/singleton input. Source order is unchanged.
    public static T[] Sort<T>(T[] input, Comparison<T> compare, Counts counts)
    {
        ArgumentNullException.ThrowIfNull(input);
        ArgumentNullException.ThrowIfNull(compare);
        ArgumentNullException.ThrowIfNull(counts);
        counts.Comparisons=counts.Writes=counts.Calls=0;counts.PeakFrames=0;
        T[] values=(T[])input.Clone();
        T[] buffer=new T[values.Length];
        SortRange(0,values.Length,1);
        return values;

        void SortRange(int start,int end,int depth)
        {
            counts.Calls++;counts.PeakFrames=Math.Max(counts.PeakFrames,depth);
            if(end-start<=1)return;
            int middle=start+(end-start)/2;
            SortRange(start,middle,depth+1);
            SortRange(middle,end,depth+1);
            int i=start,j=middle,k=start;
            while(i<middle && j<end)
            {
                counts.Comparisons++;
                buffer[k++]=compare(values[i],values[j])<=0?values[i++]:values[j++];
                counts.Writes++;
            }
            while(i<middle){buffer[k++]=values[i++];counts.Writes++;}
            while(j<end){buffer[k++]=values[j++];counts.Writes++;}
            for(int p=start;p<end;p++){values[p]=buffer[p];counts.Writes++;}
        }
    }
}


The output is six `verified` lines, one per case. Each line passed three checks: the sorted result matches an independent reference, the input array is unchanged, and the returned array is a different object. The last case shows that `int.MinValue` and `int.MaxValue` sort correctly because the comparison uses `CompareTo` rather than subtraction.

**Common mistake.** Asserting `actual.SequenceEqual(actual.OrderBy(x => x))`, which compares the output with a sorted copy of itself. That passes for any sorted output, including an empty one; the reference must be built from the *input*, as `before.OrderBy` is.


```{index} Compare Measured Operations
```

## Compare Measured Operations




```{index} Compare Measured Operations
```

Count actual key comparisons and element writes. The asymptotic expression n log₂ n is a growth model, not an exact comparison count. Different orders can exhaust one merge side earlier. Buffer copy-back contributes writes even after comparisons stop.

In [ ]:
using System;
using System.Linq;

foreach(int n in new[]{4,8,16,32})
{
 var counts=new StableSort.Counts();int[] values=Enumerable.Range(0,n).Reverse().ToArray();
 StableSort.Sort(values,(a,b)=>a.CompareTo(b),counts);
 Console.WriteLine($"n={n}, comparisons={counts.Comparisons}, writes={counts.Writes}, peak={counts.PeakFrames}, nlogn={n*Math.Log2(n):F0}");
}
public static class StableSort
{
    public sealed class Counts
    {
        public long Comparisons, Writes, Calls;
        public int PeakFrames;
    }
    // Inputs must already be sorted under compare. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, Comparison<T> compare)
    {
        ArgumentNullException.ThrowIfNull(left);
        ArgumentNullException.ThrowIfNull(right);
        ArgumentNullException.ThrowIfNull(compare);
        T[] result = new T[checked(left.Length + right.Length)];
        int i=0,j=0,k=0;
        while(i<left.Length && j<right.Length)
            result[k++]=compare(left[i],right[j])<=0?left[i++]:right[j++];
        while(i<left.Length)result[k++]=left[i++];
        while(j<right.Length)result[k++]=right[j++];
        return result;
    }
    // Fresh array even for empty/singleton input. Source order is unchanged.
    public static T[] Sort<T>(T[] input, Comparison<T> compare, Counts counts)
    {
        ArgumentNullException.ThrowIfNull(input);
        ArgumentNullException.ThrowIfNull(compare);
        ArgumentNullException.ThrowIfNull(counts);
        counts.Comparisons=counts.Writes=counts.Calls=0;counts.PeakFrames=0;
        T[] values=(T[])input.Clone();
        T[] buffer=new T[values.Length];
        SortRange(0,values.Length,1);
        return values;

        void SortRange(int start,int end,int depth)
        {
            counts.Calls++;counts.PeakFrames=Math.Max(counts.PeakFrames,depth);
            if(end-start<=1)return;
            int middle=start+(end-start)/2;
            SortRange(start,middle,depth+1);
            SortRange(middle,end,depth+1);
            int i=start,j=middle,k=start;
            while(i<middle && j<end)
            {
                counts.Comparisons++;
                buffer[k++]=compare(values[i],values[j])<=0?values[i++]:values[j++];
                counts.Writes++;
            }
            while(i<middle){buffer[k++]=values[i++];counts.Writes++;}
            while(j<end){buffer[k++]=values[j++];counts.Writes++;}
            for(int p=start;p<end;p++){values[p]=buffer[p];counts.Writes++;}
        }
    }
}


For powers of two, this implementation performs exactly 2n log₂ n counted writes: one buffer write and one copy-back write per item per internal level. Clone and buffer allocation are setup and are excluded from that counter, but included in the algorithm’s overall cost. Recursion has 2n−1 calls for nonempty input. Time remains Θ(n log n); these are deterministic operation counts, not timings.

**Common mistake.** Reading `n log₂ n` as the exact comparison count and treating the measured 17 or 49 as a bug. The model bounds growth; actual comparisons depend on when each merge side runs out, and the writes column grows exactly as `2n log₂ n` because every item is written twice per level.


```{index} Exercise
```

## Practice: Verify Merge Results




```{index} Exercise
```

Sort labeled invoices by amount in descending order while preserving arrival order for ties. Test int.MinValue and int.MaxValue safely. Reverse the comparison operands; do not subtract keys.

In [ ]:
using System;
using System.Linq;

var invoices=new[]{("A",3),("B",int.MinValue),("C",3),("D",int.MaxValue)};
// Your code starts here.
/* TODO */
// Your code ends here.

In [1]:
using System;
using System.Linq;

// Solution
var invoices=new[]{("A",3),("B",int.MinValue),("C",3),("D",int.MaxValue)};
var sorted=StableSort.Sort(invoices,(a,b)=>b.Item2.CompareTo(a.Item2),new StableSort.Counts());
Console.WriteLine(string.Join(", ",sorted.Select(x=>x.Item1)));
if(!sorted.Select(x=>x.Item1).SequenceEqual(new[]{"D","A","C","B"}))throw new Exception("Descending stability failed.");
public static class StableSort
{
    public sealed class Counts
    {
        public long Comparisons, Writes, Calls;
        public int PeakFrames;
    }
    // Inputs must already be sorted under compare. Equal left items win ties.
    public static T[] Merge<T>(T[] left, T[] right, Comparison<T> compare)
    {
        ArgumentNullException.ThrowIfNull(left);
        ArgumentNullException.ThrowIfNull(right);
        ArgumentNullException.ThrowIfNull(compare);
        T[] result = new T[checked(left.Length + right.Length)];
        int i=0,j=0,k=0;
        while(i<left.Length && j<right.Length)
            result[k++]=compare(left[i],right[j])<=0?left[i++]:right[j++];
        while(i<left.Length)result[k++]=left[i++];
        while(j<right.Length)result[k++]=right[j++];
        return result;
    }
    // Fresh array even for empty/singleton input. Source order is unchanged.
    public static T[] Sort<T>(T[] input, Comparison<T> compare, Counts counts)
    {
        ArgumentNullException.ThrowIfNull(input);
        ArgumentNullException.ThrowIfNull(compare);
        ArgumentNullException.ThrowIfNull(counts);
        counts.Comparisons=counts.Writes=counts.Calls=0;counts.PeakFrames=0;
        T[] values=(T[])input.Clone();
        T[] buffer=new T[values.Length];
        SortRange(0,values.Length,1);
        return values;

        void SortRange(int start,int end,int depth)
        {
            counts.Calls++;counts.PeakFrames=Math.Max(counts.PeakFrames,depth);
            if(end-start<=1)return;
            int middle=start+(end-start)/2;
            SortRange(start,middle,depth+1);
            SortRange(middle,end,depth+1);
            int i=start,j=middle,k=start;
            while(i<middle && j<end)
            {
                counts.Comparisons++;
                buffer[k++]=compare(values[i],values[j])<=0?values[i++]:values[j++];
                counts.Writes++;
            }
            while(i<middle){buffer[k++]=values[i++];counts.Writes++;}
            while(j<end){buffer[k++]=values[j++];counts.Writes++;}
            for(int p=start;p<end;p++){values[p]=buffer[p];counts.Writes++;}
        }
    }
}


D, A, C, B


The output is `D, A, C, B`: amounts in descending order, with A before C because they tie at 3 and A arrived first. Reversing the comparison operands produced the descending order without touching the tie rule, which is why stability survived.


```{rubric} Footnotes
```


[^2202_divide_conquer-1]: Divide-and-conquer is not automatically faster than iteration. It pays off when the split and combine structure reduces work or clarifies a naturally recursive problem.
[^2202_divide_conquer-2]: The range convention `[start, end)` includes `start` and excludes `end`. It is common in C# because the length of the range is `end - start`.


[^2203_merge_sort_lab-1]: This implementation sorts a copy using one reusable merge buffer. It does not claim in-place sorting or constant auxiliary memory.
[^2203_merge_sort_lab-2]: Stability depends on a consistent key comparison and left-first handling of equal keys at every merge.

[^development-reading]: Further reading: [Nakov and colleagues, Methods](https://introprogramming.info/english-intro-csharp-book/read-online/chapter-9-methods/). The worked business scenarios and guided reasoning here are original teaching examples.
